<div style="text-align: justify; max-width: 90ch">

# Lecture 01f: pydantic schemas and hosted inference

**Status: Optional / career-track.**

Two things production code does that the mandatory notebooks skipped:

- The JSON schema dictionaries of `lec_01c` and `lec_01d` are usually written as **pydantic models**, Python classes that generate the schema and check the answer against it.
- The one-line swap from `lec_01d` section 4: the same chain against a **hosted** model, through the OpenAI-compatible API that Ollama speaks too.

**What this notebook covers**

- A pydantic model as a typed description of an answer, and `with_structured_output` returning checked objects.
- The OpenAI-compatible API: one client, many providers, Ollama included.
- Hosted inference: what a provider sells, and where its API key goes.
- Swapping the chat model in a LangChain chain: the code stays the same; cost, privacy and latency change.
- When the local model is enough, and when hosted pays.

</div>

In [1]:
import os
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-basics"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. A schema as a class

In `lec_01c` and `lec_01d` the JSON schema was a dictionary typed by hand, and the answer came back as a plain dict. Two problems remain as a system grows:

- **Nothing checks the answer.** `format=` constrains what Ollama generates, but not every provider has such a mode, and a dict schema's answer reaches your code unchecked:
  a wrong value or a missing key surfaces later, as a `KeyError` three cells down.
- **The schema is long and hard to read**: a nested dictionary for three fields.

[**pydantic**](https://pydantic.dev/docs/validation/latest/get-started/) is the Python library for both. You describe the answer once, in a few lines;
it generates the schema and [**validates**](https://pydantic.dev/docs/validation/latest/concepts/models/) the data: it builds an object only when every field has the declared type and an allowed value.

</div>

<div style="text-align: justify; max-width: 90ch">

A **class** is a named template for records: it lists the fields every record has and the type of each. You have used objects made from classes since `lec_01a` (a `DataFrame`, an `ollama.Client`);
here you write a class. Mapped onto a table you know:

| pydantic | In a DataFrame |
| --- | --- |
| the class `ForumTag` | the column names and their dtypes |
| an object, `tag` | one row |
| a field, `tag.category` | one cell of that row |

</div>

<div style="text-align: justify; max-width: 90ch">

Read the next cell line by line:

- `class ForumTag(BaseModel):` names the template; `BaseModel` in the brackets makes it a pydantic model, with pydantic's methods such as `model_json_schema()`.
- The text in triple quotes is the docstring, a description, like those of the functions in `lec_01c`.
- `category: Literal[...]`: the part after the colon is a [**type hint**](https://docs.python.org/3/glossary.html#term-type-hint),
  the type the field holds. [`Literal`](https://docs.python.org/3/library/typing.html#typing.Literal) means "exactly one of these values".
- `needs_code: bool` and `summary: str`: a `True`/`False` and a text.
- `= Field(description=...)` adds a description to the schema. Despite the `=`, it is not a default value: the field stays required.

Plain Python ignores type hints; pydantic reads them and enforces them.

</div>

In [3]:
class ForumTag(BaseModel):
    """How a forum question is filed."""

    category: Literal["pandas", "plots", "sklearn", "python-basics", "other"]
    needs_code: bool = Field(description="whether the answer needs a code example")
    summary: str = Field(description="the question in at most ten words")


ForumTag.model_json_schema()

{'description': 'How a forum question is filed.',
 'properties': {'category': {'enum': ['pandas',
    'plots',
    'sklearn',
    'python-basics',
    'other'],
   'title': 'Category',
   'type': 'string'},
  'needs_code': {'description': 'whether the answer needs a code example',
   'title': 'Needs Code',
   'type': 'boolean'},
  'summary': {'description': 'the question in at most ten words',
   'title': 'Summary',
   'type': 'string'}},
 'required': ['category', 'needs_code', 'summary'],
 'title': 'ForumTag',
 'type': 'object'}

<div style="text-align: justify; max-width: 90ch">

That is the dictionary you typed by hand in `lec_01c`, generated from three annotated lines:

| In the class | In the schema |
| --- | --- |
| the docstring | the object's `description` |
| `Literal[...]` | `enum` |
| `bool`, `str` | `"type": "boolean"`, `"type": "string"` |
| `Field(description=...)` | the field's `description` |
| every field (none has a default) | listed in `required` |

pydantic also gives each field a `title` made from its name (`Needs Code`); it is harmless.

</div>

<div style="text-align: justify; max-width: 90ch">

The `enum` and the descriptions do different jobs. Ollama enforces the `enum` while it generates (`lec_01c`), so the category is always one of the five.
It does not put the schema into the prompt, so the model never reads the descriptions; Ollama's [docs](https://docs.ollama.com/capabilities/structured-outputs) advise repeating the schema in the prompt.
The model does see the field names, because it writes them.

```text
WRONG: category: str = Field(description="one of pandas, plots, sklearn, ...")
RIGHT: category: Literal["pandas", "plots", "sklearn", "python-basics", "other"]
```

Allowed values go in a `Literal`; instructions go in the prompt.

</div>

<div style="text-align: justify; max-width: 90ch">

LangChain accepts the class itself. `with_structured_output(ForumTag, method="json_schema")` (`lec_01d` section 3) sends the generated schema as `format=` and validates the answer into a `ForumTag` object.
Given a dict schema, as in `lec_01d`, it returns the dict unchecked.

</div>

In [4]:
llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=150,
)

tag_prompt = ChatPromptTemplate.from_template(
    "Tag this Python-course forum question.\n\nQuestion: {question}"
)

tag_chain = tag_prompt | llm.with_structured_output(ForumTag, method="json_schema")

tag = tag_chain.invoke({"question": "How do I merge two DataFrames on a customer id?"})

print(type(tag).__name__, "->", tag.category, "|", tag.needs_code, "|", tag.summary)

ForumTag -> pandas | True | Merge two DataFrames on a customer ID using pandas


<div style="text-align: justify; max-width: 90ch">

The chain returned a `ForumTag` object, not a dict. Check that the category fits a merge question (`pandas`); whatever the model chose, it is one of the five.
Fields are read with a dot, `tag.category` instead of `tag["category"]`, and an editor can offer the three names as you type `tag.`.

You have read pydantic objects since `lec_01a`: the `response` of the `ollama` client, LangChain's `AIMessage` and the `completion` of section 2 are all pydantic models,
read the same way: `response.message.content`.

</div>

<div style="text-align: justify; max-width: 90ch">

Validation earns its place when nothing constrained the output: another provider, or text you parse yourself. `ForumTag.model_validate_json(text)` runs the chain's check on a JSON string.
Given the valid JSON `{"category": "Data Manipulation", "needs_code": "maybe"}`, it raises a `ValidationError` that lists three problems:

- **`category`**: not one of the five allowed values.
- **`needs_code`**: `"maybe"` is not a boolean.
- **`summary`**: missing.

It also converts what it safely can: `"true"` becomes `True`. Inside a chain, LangChain wraps the error in an `OutputParserException`:
the call fails at the model's step, instead of a wrong value travelling on into your DataFrame.

</div>

<div style="text-align: justify; max-width: 90ch">

- **When the dict schema is enough**, all three hold:
  - one call
  - a provider that enforces the schema
  - the answer goes straight into a DataFrame, as in `lec_01c`
- **When pydantic pays**, any one holds:
  - the answer travels on through your code
  - the provider has no schema mode
  - the same record shape is used in several places

For pandas, `tag.model_dump()` turns the object back into a dict.

The cost is small: a new syntax, and calls that now fail loudly on a bad answer, which is the point. pydantic is already installed, because `ollama`, `openai`, LangChain and MLflow depend on it.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. The OpenAI-compatible API

`lec_01d` section 4 called a provider **OpenAI-compatible** when it accepts the request format of OpenAI's API; Ollama serves that format at `http://localhost:11434/v1`.
You do not need LangChain to use it: [`openai`](https://github.com/openai/openai-python), OpenAI's own Python client, talks to every such provider.
Set `base_url` to the provider's **endpoint**, the web address where its API answers, and pass its key: one client, many providers.

Against Ollama the key is any non-empty string: the client requires one and Ollama ignores it ([docs](https://docs.ollama.com/api/openai-compatibility)).

</div>

<div style="text-align: justify; max-width: 90ch">

The request and the reply use OpenAI's names for things you know from `lec_01a` and `lec_01c`:

| `ollama` client | `openai` client (next cell) |
| --- | --- |
| `client.chat(...)` | `local.chat.completions.create(...)` |
| `options={"temperature": 0}` | `temperature=0` |
| `num_predict` | `max_tokens` |
| `think=False` | `reasoning_effort="none"` |
| `response.message.content` | `completion.choices[0].message.content` |
| `prompt_eval_count`, `eval_count` | `usage.prompt_tokens`, `usage.completion_tokens` |

`choices` is a list because the format can return several answers; you asked for one. `/no_think` at the end of the question is Qwen3's own switch,
written into the prompt ([model card](https://huggingface.co/Qwen/Qwen3-1.7B)); `reasoning_effort` is the request's switch.

</div>

In [5]:
local = OpenAI(base_url=f"{OLLAMA_HOST}/v1", api_key="ollama")

completion = local.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is a DataFrame? /no_think"}
    ],
    temperature=0,
    max_tokens=80,
    reasoning_effort="none",  # Ollama maps this to think=False
)

print(completion.choices[0].message.content)
print(completion.usage)

A DataFrame is a two-dimensional data structure in Python, commonly used in pandas, that stores data in rows and columns, allowing for efficient manipulation and analysis of data.
CompletionUsage(completion_tokens=34, prompt_tokens=29, total_tokens=63, completion_tokens_details=None, prompt_tokens_details=None)


<div style="text-align: justify; max-width: 90ch">

`usage` lists the tokens in (`prompt_tokens`) and out (`completion_tokens`): the `prompt_eval_count` and `eval_count` of `lec_01a`, under OpenAI's names.
Locally those tokens cost time; a hosted provider bills them (`lec_01a` section 3).

</div>

<div style="text-align: justify; max-width: 90ch">

**Hosted inference** means running the model on someone else's hardware: your code sends each request over the internet to a **hosted provider** (guide `01b_llmfit`) and gets the answer back.
It solves what a laptop cannot:

- **Bigger models.** Models with hundreds of billions of parameters, far beyond any laptop's RAM (`lec_01b` section 4).
- **Speed.** Data-centre GPUs, much faster than the tokens per second you measured in `lec_01a`.
- **No setup.** No download, no RAM or GPU of your own; any computer with internet access can call it.

It costs:

- money per token;
- your data, sent to another company;
- your independence: your system depends on that company.

Section 4 weighs the two.

</div>

<div style="text-align: justify; max-width: 90ch">

A provider knows who is calling from the API key (`lec_01a`), a secret string sent with every request; it bills the account the key belongs to.
Whoever has your key spends your money, so it gets the Hugging Face token's treatment from guide `01d_env_hugging_face`:
in `.env`, never in a notebook, replaced at once if it leaks.

The same format, a different `base_url`, and a key from a provider account, is a hosted model. The cell below runs only if you put a provider's endpoint, key and model in `.env`:

```text
HOSTED_BASE_URL=https://api.example.com/v1
HOSTED_API_KEY=...
HOSTED_MODEL=...
```

</div>

<div style="text-align: justify; max-width: 90ch">

Any OpenAI-compatible provider works. [Hugging Face Inference Providers](https://huggingface.co/docs/inference-providers/index) is one:
a **router**, a single address, `https://router.huggingface.co/v1`, that forwards each request to one of several partner companies running the model and bills your Hugging Face account.

- **Key:** your `HF_TOKEN`; a read token covers inference ([token roles](https://huggingface.co/docs/hub/security-tokens)).
- **Model:** a Hub model id, for example `openai/gpt-oss-120b` from their getting-started example.
- **Price:** every account gets a small monthly credit; beyond it you buy credits ([pricing](https://huggingface.co/docs/inference-providers/pricing)).

</div>

In [6]:
HOSTED_BASE_URL = os.environ.get("HOSTED_BASE_URL")
HOSTED_API_KEY = os.environ.get("HOSTED_API_KEY")
HOSTED_MODEL = os.environ.get("HOSTED_MODEL")

if HOSTED_BASE_URL and HOSTED_API_KEY and HOSTED_MODEL:
    hosted = OpenAI(base_url=HOSTED_BASE_URL, api_key=HOSTED_API_KEY)

    completion = hosted.chat.completions.create(
        model=HOSTED_MODEL,
        messages=[{"role": "user", "content": "In one sentence, what is a DataFrame?"}],
        temperature=0,
        max_tokens=80,
    )

    print(completion.choices[0].message.content)
    print(completion.usage)
else:
    print("No hosted endpoint configured in .env; skipping (this is fine).")

No hosted endpoint configured in .env; skipping (this is fine).


<div style="text-align: justify; max-width: 90ch">

## 3. The swap inside a chain

`ChatOpenAI`, from the `langchain-openai` partner package (`lec_01d` section 4), is LangChain's chat model for any OpenAI-compatible provider,
and takes the `openai` client's argument names from section 2. The cell below builds it for the hosted provider when `.env` names one, and for Ollama's `/v1` address otherwise.
The prompt, the parser and the `|` are the same as in `lec_01d`.

</div>

In [7]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a teaching assistant for a Python course. Answer in one sentence.",
        ),
        ("human", "{question}"),
    ]
)

if HOSTED_BASE_URL and HOSTED_API_KEY and HOSTED_MODEL:
    swapped_llm = ChatOpenAI(
        model=HOSTED_MODEL,
        base_url=HOSTED_BASE_URL,
        api_key=HOSTED_API_KEY,
        temperature=0,
    )
    print("using the hosted model")
else:
    swapped_llm = ChatOpenAI(
        model=CHAT_MODEL,
        base_url=f"{OLLAMA_HOST}/v1",
        api_key="ollama",
        temperature=0,
        reasoning_effort="none",
    )
    print("no hosted endpoint; using Ollama through its OpenAI-compatible API")

chain = prompt | swapped_llm | StrOutputParser()

print(chain.invoke({"question": "What does random_state do?"}))

no hosted endpoint; using Ollama through its OpenAI-compatible API


The `random_state` parameter in scikit-learn is used to set the random seed for the random number generator, ensuring reproducibility of results.


<div style="text-align: justify; max-width: 90ch">

If the cell printed "no hosted endpoint", the chain ran on Ollama, through `ChatOpenAI` and the `/v1` address: the code path a hosted model takes, with nothing leaving the laptop.
Check the answer against what you know: `random_state` fixes the seed so results repeat. With a key in `.env`, the same cell sends the question to the hosted model instead.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Local or hosted

The code did not change; the requirements table of `lec_01b` reads differently:

| Requirement | Local (Ollama) | Hosted |
| --- | --- | --- |
| **Privacy** | nothing leaves the laptop | every question reaches the provider, and through a router a second company |
| **Cost** | hardware you already own | per token in and out (`usage`) |
| **Latency** | your tokens/s | a network round trip, then a much faster machine |
| **Model size** | what fits your RAM | models no laptop can hold |
| **Maintenance** | you decide when a model changes | the provider can change, limit or retire it |
| **Availability** | works offline | needs the internet and the provider to be up |

</div>

<div style="text-align: justify; max-width: 90ch">

- **Local is enough** for private data, many cheap calls (tagging every forum question), offline use, or when a small model meets the requirements,
  as `qwen3:1.7b` does for this course.
- **Hosted pays** for a task the small model fails (Greek, long documents, hard reasoning), a few calls a day that do not justify hardware,
  or users without a capable computer.
- **Both** in one system: a local model for private or frequent calls, a hosted one for the hard cases. Because the swap is one line, a requirement can change without a rewrite.

Before sending real data, read the provider's data policy: Hugging Face's router, for example, does not store requests and points to each partner's own policy ([security](https://huggingface.co/docs/inference-providers/security)).

</div>

<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Take the course's own assistant, the requirements table of `lec_01b` section 1. Which rows would a hosted model meet better, and which would it fail?
Then name a job, from your work or studies, where the answer flips.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A pydantic `BaseModel` is a typed schema: a class whose type hints generate the JSON schema and **validate** the answer into an object.
- Allowed values go in a `Literal`; Ollama does not show the model the descriptions, so instructions go in the prompt.
- The OpenAI-compatible API is the common format; Ollama serves it at `/v1`, so the `openai` client and `ChatOpenAI` work locally and hosted alike.
- A hosted provider sells bigger, faster models for money per token, your data leaving the laptop, and a dependency; its API key lives in `.env`.
- Swapping the model is one line; what changes is the requirements table, not the chain.

Back to the mandatory path: **lecture 2**.

</div>